# Data Processing

Processes raw files and creates a nice csv file

In [1]:
import pandas as pd
import numpy as np
from os import listdir
from pathlib import Path

In [2]:
RAW_DATA_DIR = Path.cwd().parents[2] / Path("data/raw/")
RAW_DATA_DIR

WindowsPath('c:/Users/chris/Desktop/MyCode/ethnohack_2026/Top-secret-project/data/raw')

In [3]:
raw_filenames = [RAW_DATA_DIR / Path(f) for f in listdir(RAW_DATA_DIR) if Path(f).suffix == ".xlsx"]

In [4]:
df = pd.read_excel(raw_filenames[0], sheet_name=9, skiprows=5)
df

,Code ATC5,Libellé ATC5,Base de remboursement 2021-01,Nombre de boites remboursées \n2021-01,Montant remboursé \n2021-01,Base de remboursement 2021-02,Nombre de boites remboursées \n2021-02,Montant remboursé \n2021-02,Base de remboursement 2021-03,Nombre de boites remboursées \n2021-03,Montant remboursé \n2021-03,Base de remboursement 2021-04,Nombre de boites remboursées \n2021-04,Montant remboursé \n2021-04,Base de remboursement 2021-05,Nombre de boites remboursées \n2021-05,Montant remboursé \n2021-05,Base de remboursement 2021-06,Nombre de boites remboursées \n2021-06,Montant remboursé \n2021-06
0,A01AA01,SODIUM FLUORURE,6.233800e+03,3732,2.068914e+03,6.263680e+03,3781,2.095821e+03,7.138570e+03,4268,2.375731e+03,6.247790e+03,3706,2.088466e+03,5.482440e+03,3215,1.831829e+03,6.147300e+03,3637,2.093666e+03
1,A01AA02,SODIUM MONOFLUOROPHOSPHATE,5.546068e+04,5164,4.269829e+04,5.417906e+04,5071,4.114861e+04,6.317682e+04,5918,4.774601e+04,5.795225e+04,5429,4.423414e+04,5.527642e+04,5181,4.188342e+04,6.156594e+04,5771,4.651061e+04
2,A01AA51,SODIUM FLUORURE EN ASSOCIATION,7.751732e+04,29201,2.370117e+04,7.692961e+04,28948,2.343427e+04,8.436429e+04,31751,2.583929e+04,7.297094e+04,27456,2.229768e+04,6.320032e+04,23755,1.927660e+04,7.181397e+04,27009,2.198152e+04
3,A01AB03,CHLORHEXIDINE,1.439890e+06,561367,2.893059e+05,1.637991e+06,622723,3.229237e+05,1.928634e+06,725890,3.817712e+05,1.777838e+06,669520,3.510056e+05,1.534571e+06,575851,3.061034e+05,1.873613e+06,702182,3.700958e+05
4,A01AB09,MICONAZOLE,2.955899e+05,43539,2.140734e+05,2.970149e+05,43828,2.146263e+05,3.508785e+05,51445,2.543139e+05,3.189760e+05,46867,2.319869e+05,2.805541e+05,41069,2.038459e+05,3.379457e+05,49693,2.455082e+05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1185,V08CA08,GADOBENIQUE ACIDE,1.240735e+05,2589,9.210719e+04,1.185995e+05,2468,8.739619e+04,1.397384e+05,2960,1.020834e+05,1.312110e+05,2775,9.615663e+04,1.168729e+05,2475,8.582647e+04,1.388920e+05,2950,1.019573e+05
1186,V08CA09,GADOBUTROL,2.547547e+06,36013,1.817326e+06,2.519077e+06,35844,1.789359e+06,2.870168e+06,40815,2.036758e+06,2.619485e+06,37056,1.855198e+06,2.375070e+06,33612,1.686153e+06,2.809371e+06,39578,1.993789e+06
1187,V08DA05,SOUFRE HEXAFLUORURE,1.033440e+05,1561,8.011790e+04,9.057303e+04,1374,7.146701e+04,1.041350e+05,1580,8.082727e+04,8.842194e+04,1342,6.859481e+04,8.317516e+04,1262,6.456878e+04,9.816779e+04,1491,7.593679e+04
1188,W,Homéopathie,3.892307e+06,1725254,4.631871e+05,2.192165e+06,385884,4.656791e+04,2.336605e+06,310490,1.510890e+04,2.120927e+06,259530,7.499013e+03,1.848019e+06,217293,5.429830e+03,2.185423e+06,254509,4.868405e+03


In [5]:
INDEX_COLS = ["ATC5", "full_name"]

def clean_df(df: pd.DataFrame) -> pd.DataFrame:
    # basic cleaning
    df.columns = df.columns.str.replace('\n', '', regex=False)
    df.columns = df.columns.str.strip().str.lower()
    df = df.dropna(how='all')

    # numeric restoration
    numeric_cols = df.columns[2:]
    for col in numeric_cols:
        # 'coerce' converts non numeric values to NaN
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # use only number of boxes
    columns_to_drop = [col for col in numeric_cols if not col.startswith("nombre")]
    df = df.drop(columns=columns_to_drop)
    df.columns = INDEX_COLS + [
        col.replace("nombre de boites remboursées", "").strip() 
        for col in df.columns[2:]
    ]

    # drop the last 2 rows that contain useless information for the project
    df = df.iloc[:-2, :]
    return df

In [6]:
clean_df(df)

,ATC5,full_name,2021-01,2021-02,2021-03,2021-04,2021-05,2021-06
0,A01AA01,SODIUM FLUORURE,3732,3781,4268,3706,3215,3637
1,A01AA02,SODIUM MONOFLUOROPHOSPHATE,5164,5071,5918,5429,5181,5771
2,A01AA51,SODIUM FLUORURE EN ASSOCIATION,29201,28948,31751,27456,23755,27009
3,A01AB03,CHLORHEXIDINE,561367,622723,725890,669520,575851,702182
4,A01AB09,MICONAZOLE,43539,43828,51445,46867,41069,49693
...,...,...,...,...,...,...,...,...
1183,V08CA03,GADODIAMIDE,2,1,0,0,0,0
1184,V08CA04,GADOTERIDOL,18351,18214,21472,20165,17739,20292
1185,V08CA08,GADOBENIQUE ACIDE,2589,2468,2960,2775,2475,2950
1186,V08CA09,GADOBUTROL,36013,35844,40815,37056,33612,39578


In [ ]:
dfs = []

for f in raw_filenames:
    cur_df = pd.read_excel(f, sheet_name=9, skiprows=5)
    cur_df = clean_df(cur_df)
    dfs.append(cur_df.set_index(INDEX_COLS))

total_df = pd.concat(dfs, axis=1).reset_index()
total_df

C:\Users\chris\AppData\Local\Temp\ipykernel_27272\645690869.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  total_df = pd.concat(dfs, axis=1).reset_index()


,ATC5,full_name,2021-01,2021-02,2021-03,2021-04,2021-05,2021-06,2021-07,2021-08,...,2020-03,2020-04,2020-05,2020-06,2020-07,2020-08,2020-09,2020-10,2020-11,2020-12
0,A01AA01,SODIUM FLUORURE,3732.0,3781.0,4268.0,3706.0,3215.0,3637.0,3444.0,2744.0,...,4564.0,2998.0,3180.0,4565.0,4256.0,3109.0,4494.0,4528.0,4184.0,4288.0
1,A01AA02,SODIUM MONOFLUOROPHOSPHATE,5164.0,5071.0,5918.0,5429.0,5181.0,5771.0,5363.0,4791.0,...,6143.0,4213.0,4439.0,5725.0,5671.0,4551.0,5715.0,5560.0,5321.0,5927.0
2,A01AA51,SODIUM FLUORURE EN ASSOCIATION,29201.0,28948.0,31751.0,27456.0,23755.0,27009.0,24081.0,20067.0,...,32536.0,23813.0,26910.0,31441.0,30202.0,22436.0,33086.0,31112.0,31155.0,31154.0
3,A01AB03,CHLORHEXIDINE,561367.0,622723.0,725890.0,669520.0,575851.0,702182.0,645230.0,493421.0,...,463746.0,271955.0,331285.0,503911.0,644864.0,466040.0,645615.0,644503.0,621399.0,652279.0
4,A01AB09,MICONAZOLE,43539.0,43828.0,51445.0,46867.0,41069.0,49693.0,46993.0,42808.0,...,61309.0,50034.0,49061.0,55976.0,52733.0,42036.0,50615.0,48729.0,49981.0,51137.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1783,L01XE39,MIDOSTAURINE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,92.0
1784,L01XE44,LORLATINIB,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1.0,189.0,261.0,246.0,257.0,292.0
1785,L01XE53,LAROTRECTINIB,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,7.0
1786,L01XX,AUTRES ANTINEOPLASIQUES,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,682.0,595.0,640.0,656.0,678.0,701.0
